# Model test & validation: Polynomial Regression (deg 2 & 4, Ridge)

Tests both polynomial degrees together since they share the same failure modes. Key question: does higher degree (4) overfit/destabilize more than degree 2, especially under extrapolation?

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

def mape(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
def r2(y_true, y_pred):
    return float(r2_score(y_true, y_pred))

df_raw = pd.read_csv("../../data/chf_long_clean.csv")
df = df_raw[df_raw.X != 1.0].reset_index(drop=True)
FEATURES = ["P", "G", "X"]
TARGET = "CHF"
sorted_P = sorted(df.P.unique())

# Split A (random, seed 0) -- quick interpolation check
X_all, y_all = df[FEATURES].values, df[TARGET].values
XtrA, XteA, ytrA, yteA = train_test_split(X_all, y_all, test_size=0.2, random_state=0)

# Split C (edge extrapolation) -- the honest test
train_dfC = df[df.P <= 16000].reset_index(drop=True)
test_dfC = df[df.P >= 17000].reset_index(drop=True)
XtrC, ytrC = train_dfC[FEATURES].values, train_dfC[TARGET].values
XteC, yteC = test_dfC[FEATURES].values, test_dfC[TARGET].values

print(f"Split A: {len(XtrA)} train / {len(XteA)} test")
print(f"Split C: {len(XtrC)} train / {len(XteC)} test")


## Fit on Split A (interpolation) and Split C (extrapolation)

In [ ]:

results = {}
for degree in [2, 4]:
    scaler = StandardScaler().fit(XtrA)
    model_A = make_pipeline(PolynomialFeatures(degree=degree), Ridge(alpha=1.0))
    model_A.fit(scaler.transform(XtrA), np.log(ytrA))
    predA = np.exp(model_A.predict(scaler.transform(XteA)))

    scalerC = StandardScaler().fit(XtrC)
    model_C = make_pipeline(PolynomialFeatures(degree=degree), Ridge(alpha=1.0))
    model_C.fit(scalerC.transform(XtrC), np.log(ytrC))
    predC = np.exp(model_C.predict(scalerC.transform(XteC)))

    results[degree] = dict(model_A=model_A, model_C=model_C, predA=predA, predC=predC, scalerC=scalerC)
    print(f"Degree {degree} -- Split A: R2={r2(yteA,predA):.4f} | Split C: R2={r2(yteC,predC):.4f}")


## Edge-case tests

Check how many polynomial terms each degree generates (feature-count blowup), and whether Ridge coefficient magnitudes grow with degree (a sign of extrapolation instability -- large coefficients on high-degree terms can produce wild predictions far from the training distribution's typical scale).

In [ ]:

for degree in [2, 4]:
    poly = results[degree]["model_A"].named_steps["polynomialfeatures"]
    ridge = results[degree]["model_A"].named_steps["ridge"]
    n_terms = poly.n_output_features_
    max_coef = np.abs(ridge.coef_).max()
    print(f"Degree {degree}: {n_terms} polynomial terms, max |coefficient|={max_coef:.3f}")

print("\nExpectation: degree 4 has far more terms and larger coefficient magnitudes "
      "than degree 2 -- consistent with the base notebook's finding that Poly4_Ridge "
      "(raw target) had by far the worst Split-A MAPE (96.6%) among all models, "
      "and that Poly4 fared worse than Poly2 specifically on Split C extrapolation.")
assert results[4]["model_A"].named_steps["polynomialfeatures"].n_output_features_ > \
       results[2]["model_A"].named_steps["polynomialfeatures"].n_output_features_
print("PASS: degree 4 generates strictly more terms than degree 2, as expected.")


## Diagnostic plot

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for degree, color in [(2, "tab:blue"), (4, "tab:orange")]:
    axes[0].scatter(yteA, results[degree]["predA"], s=5, alpha=0.3, color=color, label=f"deg{degree}")
    axes[1].scatter(yteC, results[degree]["predC"], s=5, alpha=0.3, color=color, label=f"deg{degree}")
for ax, y in [(axes[0], yteA), (axes[1], yteC)]:
    lims = [0, y.max()]
    ax.plot(lims, lims, "r--")
    ax.legend()
axes[0].set_title("Split A parity"); axes[1].set_title("Split C parity (extrapolation)")
for ax in axes:
    ax.set_xlabel("True CHF"); ax.set_ylabel("Predicted CHF")
plt.tight_layout()
plt.savefig("../results/model_tests_polynomial.png", dpi=100)
plt.show()
